In [2]:
import netCDF4 as nc
import numpy as np
import sys

# ================= CONFIGURATION =================
INPUT_FILE='/Users/jinyuntang/work/github/EcoSIM/input_data/ecosim_pftpar_20251224.nc'
OUTPUT_FILE = '/Users/jinyuntang/work/ibi/grass_clova_pftpar.nc'

# List the PFT names you want to extract here.
# These must match the names in the 'pfts' variable exactly.
TARGET_PFTS = [
    "gr3s32",
    "clvs32"
]

# Variables to copy entirely regardless of dimensions (as requested)
COPY_ENTIRELY = [
    'pfts_short', 
    'pfts_long', 
    'koppen_clim_no', 
    'koppen_clim_short', 
    'koppen_clim_long'
]
# =================================================

def get_pft_indices(src):
    """
    Reads the 'pfts' variable, decodes the characters to strings,
    and finds the indices of the TARGET_PFTS.
    """
    if 'pfts' not in src.variables:
        print("Error: Variable 'pfts' not found in source file.")
        sys.exit(1)

    # Read raw char array. Shape is typically (npfts, nchars1)
    pft_chars = src.variables['pfts'][:]
    
    # Convert character arrays to python strings
    # NetCDF char arrays are often masked arrays of bytes
    pft_names = []
    for row in pft_chars:
        # Join bytes, decode to string, and strip whitespace/nulls
        if isinstance(row, np.ma.MaskedArray):
            row = row.filled(b'')
        name = b"".join(row).decode('utf-8').strip()
        pft_names.append(name)

    indices = []
    found_pfts = []
    
    for target in TARGET_PFTS:
        if target in pft_names:
            idx = pft_names.index(target)
            indices.append(idx)
            found_pfts.append(target)
        else:
            print(f"Warning: Target PFT '{target}' not found in source file.")

    print(f"Found {len(indices)} of {len(TARGET_PFTS)} requested PFTs.")
    return indices

def main():
    try:
        with nc.Dataset(INPUT_FILE, 'r') as src, nc.Dataset(OUTPUT_FILE, 'w') as dst:
            print(f"Opening {INPUT_FILE}...")
            
            # 1. Get indices for extraction
            indices = get_pft_indices(src)
            if not indices:
                print("No valid PFTs found. Exiting.")
                return

            # 2. Copy Global Attributes
            print("Copying global attributes...")
            dst.setncatts(src.__dict__)

            # 3. Create Dimensions
            print("Creating dimensions...")
            for name, dimension in src.dimensions.items():
                dim_len = len(dimension)
                
                # If this is the 'npfts' dimension, we resize it to match our extracted count
                # Note: If the source is UNLIMITED, we can keep dst UNLIMITED or fix it.
                # Here we usually treat the subset as fixed or UNLIMITED based on source.
                if name == 'npfts':
                    new_len = len(indices) if not dimension.isunlimited() else None
                    dst.createDimension(name, new_len)
                    print(f"  Dimension '{name}' resized from {dim_len} to {len(indices)}")
                else:
                    new_len = dim_len if not dimension.isunlimited() else None
                    dst.createDimension(name, new_len)

            # 4. Copy Variables
            print("Copying variables...")
            for name, variable in src.variables.items():
                
                # Create the variable in the destination file
                # fill_value is handled during creation if it exists in attributes
                fill_value = variable.getncattr('_FillValue') if '_FillValue' in variable.ncattrs() else None
                dst_var = dst.createVariable(name, variable.datatype, variable.dimensions, fill_value=fill_value)
                
                # Copy variable attributes (excluding _FillValue as it's set in createVariable)
                attrs = {k: variable.getncattr(k) for k in variable.ncattrs() if k != '_FillValue'}
                dst_var.setncatts(attrs)

                # 5. Fill Data
                # logic: 
                # A. If in COPY_ENTIRELY list -> Copy all
                # B. If 'npfts' is in dimensions -> Slice based on indices
                # C. Otherwise -> Copy all
                
                if name in COPY_ENTIRELY:
                    dst_var[:] = variable[:]
                
                elif 'npfts' in variable.dimensions:
                    # Assume 'npfts' is the first dimension (standard for this file type)
                    # If npfts is not the first dimension, we need more complex slicing, 
                    # but looking at the CDL, npfts is always dim 0.
                    if variable.dimensions[0] == 'npfts':
                        dst_var[:] = variable[indices]
                    else:
                        # Fallback: if npfts is elsewhere (e.g. dim 1), slicing is trickier.
                        # For ecosim_pftpar, npfts is consistently 0.
                        print(f"  Warning: Variable {name} has npfts but not at dim 0. Copying logic may need adjustment.")
                        # Attempt generalized slicing if needed:
                        # slices = [indices if dim == 'npfts' else slice(None) for dim in variable.dimensions]
                        # dst_var[:] = variable[tuple(slices)]
                        pass 
                else:
                    # Variables that don't depend on npfts and aren't in the specific list
                    dst_var[:] = variable[:]

            print(f"Success! Extracted file saved to {OUTPUT_FILE}")

    except FileNotFoundError:
        print(f"Error: Could not find file {INPUT_FILE}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()

Opening /Users/jinyuntang/work/github/EcoSIM/input_data/ecosim_pftpar_20251224.nc...
Found 2 of 2 requested PFTs.
Copying global attributes...
Creating dimensions...
  Dimension 'npfts' resized from 77 to 2
Copying variables...
Success! Extracted file saved to /Users/jinyuntang/work/ibi/grass_clova_pftpar.nc
